# GeoSlide: Exploratory Data Analysis (EDA)
### Phase 2 — Understanding the Wireless Sensor Network (WSN) Landslide Dataset

---

## 1. Introduction

### 1.1 What is Exploratory Data Analysis (EDA)?

**Exploratory Data Analysis (EDA)** is the process of investigating a dataset in order to summarize its main
characteristics, often with the help of statistical summaries and visualizations, *before* any modeling or
formal statistical testing takes place. EDA is fundamentally about **asking questions of the data**: What does
the data look like? Are there missing values? Are there outliers? What relationships exist between variables?
Is the target variable balanced?

### 1.2 Why is EDA Important in Machine Learning?

EDA is a critical first step in any machine learning project because it:

- **Builds intuition** about the structure, quality, and distribution of the data.
- **Surfaces data quality issues** such as missing values, duplicate records, or inconsistent formatting early,
  before they silently corrupt downstream modeling.
- **Reveals relationships** between features and the target variable that can guide feature selection and
  model choice later in the project.
- **Detects class imbalance**, which is especially important in landslide prediction problems where
  "landslide" events are typically rarer than "no landslide" events.
- **Identifies potential outliers and anomalies** that may need special handling.
- **Informs preprocessing decisions** (to be carried out in a later, separate phase) by highlighting what kind
  of cleaning, transformation, or encoding might eventually be required.
- **Prevents "garbage in, garbage out"** — a model is only as good as the data used to train it, and EDA is how
  we come to trust (or distrust) our data.

### 1.3 Objective of This Notebook

This notebook performs **Phase 2: Exploratory Data Analysis** for the GeoSlide project, which aims to predict
landslide occurrence using data collected from a Wireless Sensor Network (WSN). The objective of this notebook
is strictly limited to **exploration and understanding** of the raw dataset. Specifically, this notebook will:

1. Load and inspect the raw dataset (`wsn_landslide_data.csv`).
2. Examine the structure, data types, and summary statistics of the dataset.
3. Analyze the target variable (`Label`) for class balance.
4. Identify missing values.
5. Visualize the distributions of key numerical sensor features.
6. Detect potential outliers using boxplots.
7. Study correlations between numerical features.
8. Explore relationships between individual features and the target variable.
9. Summarize the key findings that will inform later phases of the project.

> **Important scope note:** This notebook does **not** perform any data cleaning, preprocessing, encoding,
> normalization, feature engineering, dataset splitting, or model training. Those tasks belong to subsequent
> phases of the GeoSlide project and are intentionally out of scope here.


---
## 2. Import Libraries

We import the core libraries needed for data handling (`pandas`, `numpy`) and visualization
(`matplotlib`, `plotly`). Warnings are suppressed to keep the notebook output clean and readable, which is
good practice for a report-style notebook.


In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Static visualization
import matplotlib.pyplot as plt

# Interactive visualization
import plotly.express as px
import plotly.graph_objects as go

# Warnings
import warnings
warnings.filterwarnings("ignore")

# Display options for a cleaner report
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")


---
## 3. Load Dataset

We load the raw WSN landslide dataset from `datasets/training/wsn_landslide_data.csv`. This is the same raw
file that will later be used for preprocessing in subsequent phases — at this stage we only **read** it, we do
not modify it in any way.


In [ ]:
# Path to the raw dataset
DATA_PATH = "datasets/training/wsn_landslide_data.csv"

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")


In [ ]:
# Shape of the dataset (rows, columns)
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")


In [ ]:
# First 5 rows of the dataset
df.head()


In [ ]:
# Last 5 rows of the dataset
df.tail()


**Observation:** The `head()` and `tail()` views give us a first look at the raw sensor readings and the
target `Label` column. This preliminary look helps confirm that the file was read correctly and that column
names and value formats look as expected.


---
## 4. Dataset Overview

Before diving into individual features, we take a broader look at the dataset: its structure, data types, and
summary statistics. This helps us understand what kind of features we are working with (numerical vs.
categorical) and whether any columns require closer inspection.


In [ ]:
# Concise summary of the dataframe: dtypes, non-null counts, memory usage
df.info()


In [ ]:
# Descriptive statistics for numerical columns
df.describe().T


In [ ]:
# Column names
print("Columns in the dataset:")
list(df.columns)


In [ ]:
# Data types of each column
df.dtypes


**Observations:**
- `info()` tells us the total number of entries, the non-null count per column (a first hint at missing
  values, examined more formally in Section 6), and the memory footprint of the dataset.
- `describe()` summarizes the central tendency (mean, median via 50%), spread (std, min, max, quartiles) of
  each numerical column, which is useful for spotting features with unusually large ranges or suspicious
  values (e.g., negative rainfall, which would be physically implausible).
- The column names and data types confirm which columns are numerical (sensor readings) and which are
  categorical/target (e.g., `Label`).


---
## 5. Target Variable Analysis

The target variable for the GeoSlide project is the `Label` column, which indicates whether a landslide event
occurred. Understanding the balance of this column is essential — an imbalanced target is one of the most
important characteristics to be aware of before any modeling is attempted (in a later phase).


In [ ]:
# Raw counts of each class in the Label column
label_counts = df["Label"].value_counts()
label_counts


In [ ]:
# Percentage distribution of each class
label_percentages = df["Label"].value_counts(normalize=True) * 100
label_percentages.round(2)


In [ ]:
# Plotly bar chart of Label counts
fig_bar = px.bar(
    x=label_counts.index.astype(str),
    y=label_counts.values,
    labels={"x": "Label", "y": "Count"},
    title="Distribution of Target Variable (Label) - Counts",
    text=label_counts.values,
    color=label_counts.index.astype(str),
)
fig_bar.update_traces(textposition="outside")
fig_bar.update_layout(showlegend=False)
fig_bar.show()


In [ ]:
# Plotly pie chart of Label proportions
fig_pie = px.pie(
    names=label_counts.index.astype(str),
    values=label_counts.values,
    title="Distribution of Target Variable (Label) - Proportions",
    hole=0.35,
)
fig_pie.update_traces(textinfo="percent+label")
fig_pie.show()


**Observations:**
- The bar chart shows the absolute number of records belonging to each class of `Label`.
- The pie chart shows the relative proportion of each class.
- If one class strongly outnumbers the other (i.e., landslide events are much rarer than non-events, which is
  typically expected in this domain), the dataset is **imbalanced**. This is an important characteristic to
  keep in mind for later modeling phases — evaluation metrics such as accuracy can be misleading on imbalanced
  data, and techniques such as resampling or class weighting may eventually need to be considered.
- No corrective action is taken here; this section is purely observational.


---
## 6. Missing Value Analysis

Missing values can bias analysis and later modeling if not properly understood. In this section we quantify
missing data per column, both in absolute counts and as a percentage of the total dataset, and visualize the
result. No missing values are imputed or removed here — that belongs to the preprocessing phase.


In [ ]:
# Missing value counts per column
missing_counts = df.isnull().sum()
missing_counts


In [ ]:
# Missing value percentage per column
missing_percentages = (df.isnull().sum() / len(df)) * 100
missing_percentages.round(2)


In [ ]:
# Combine into a single summary dataframe, sorted by missing percentage
missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing Percentage (%)": missing_percentages.round(2)
}).sort_values(by="Missing Percentage (%)", ascending=False)

missing_summary


In [ ]:
# Plotly bar chart visualizing missing percentage per column
fig_missing = px.bar(
    missing_summary.reset_index().rename(columns={"index": "Column"}),
    x="Column",
    y="Missing Percentage (%)",
    title="Percentage of Missing Values per Column",
    text="Missing Percentage (%)",
    color="Missing Percentage (%)",
    color_continuous_scale="Reds",
)
fig_missing.update_traces(textposition="outside")
fig_missing.update_layout(xaxis_tickangle=-45)
fig_missing.show()


**Findings:**
- Columns with a missing percentage of 0% are fully populated and require no attention regarding completeness.
- Columns with a non-zero missing percentage represent gaps in the sensor data. This is common in WSN datasets
  due to sensor faults, transmission errors, or connectivity issues in the field.
- The **extent** of missingness (a few isolated readings vs. a substantial portion of a column) will determine
  the appropriate imputation or removal strategy — this decision is deferred to the preprocessing phase and is
  **not** performed here.


---
## 7. Numerical Feature Distributions

In this section we examine the distribution of the key numerical sensor features collected by the WSN:
**Rainfall, Soil Moisture, Temperature, Humidity, Slope Angle, Elevation,** and **NDVI** (Normalized Difference
Vegetation Index). If a column name differs slightly from the exact label (e.g., different casing or units in
the header), the closest matching column in the dataset is used automatically.

Histograms let us see the shape of each distribution (e.g., normal, skewed, bimodal), the typical range of
values, and any obviously unusual spikes or gaps.


In [ ]:
# Helper to find the closest matching column name for a target feature name
import difflib

target_features = [
    "Rainfall",
    "Soil Moisture",
    "Temperature",
    "Humidity",
    "Slope Angle",
    "Elevation",
    "NDVI",
]

def find_closest_column(target, columns):
    # Try exact case-insensitive match first (ignoring spaces/underscores)
    normalized_target = target.lower().replace(" ", "").replace("_", "")
    for col in columns:
        normalized_col = col.lower().replace(" ", "").replace("_", "")
        if normalized_target == normalized_col:
            return col
    # Fall back to fuzzy matching
    matches = difflib.get_close_matches(target, columns, n=1, cutoff=0.4)
    if matches:
        return matches[0]
    # Fall back to substring containment
    for col in columns:
        if normalized_target in col.lower().replace(" ", "").replace("_", ""):
            return col
    return None

feature_map = {}
for feat in target_features:
    match = find_closest_column(feat, df.columns.tolist())
    feature_map[feat] = match

print("Feature name -> Closest matching column in dataset:")
for k, v in feature_map.items():
    print(f"  {k:15s} -> {v}")


In [ ]:
# Generate a Plotly histogram for each matched numerical feature
for feat, col in feature_map.items():
    if col is None:
        print(f"No matching column found for '{feat}'. Skipping.")
        continue
    if col not in df.select_dtypes(include=[np.number]).columns:
        print(f"Column '{col}' (matched to '{feat}') is not numeric. Skipping histogram.")
        continue

    fig_hist = px.histogram(
        df,
        x=col,
        nbins=40,
        title=f"Distribution of {feat} ({col})",
        marginal="box",
        color_discrete_sequence=["#2E86AB"],
    )
    fig_hist.update_layout(bargap=0.05)
    fig_hist.show()


**Observations after distributions:**
- **Rainfall** often shows a right-skewed distribution, with many low/zero-rainfall readings and a long tail
  of high-rainfall events — these high-rainfall events are of particular interest for landslide triggering.
- **Soil Moisture** typically ranges within a bounded percentage scale; distributions clustered near the upper
  end may indicate saturated soil conditions associated with instability.
- **Temperature** and **Humidity** usually follow closer-to-normal distributions reflecting typical ambient
  environmental conditions at the sensor site.
- **Slope Angle** and **Elevation** are static/topographic features that may show multiple modes or clusters if
  sensors are deployed across multiple distinct sites.
- **NDVI** values close to 1 indicate dense vegetation, while values near 0 or negative indicate sparse
  vegetation or bare/water surfaces — this is relevant since vegetation cover affects slope stability.
- The `marginal="box"` option attached to each histogram gives an at-a-glance summary of quartiles and
  potential outliers, which is explored more formally in the next section.


---
## 8. Boxplots

Boxplots are an effective way to summarize the spread of a numerical feature and to visually flag potential
outliers (points beyond the whiskers, i.e., outside 1.5x the interquartile range). We create a boxplot for each
of the important numerical features identified above.


In [ ]:
# Generate a Plotly boxplot for each matched numerical feature
for feat, col in feature_map.items():
    if col is None or col not in df.select_dtypes(include=[np.number]).columns:
        continue

    fig_box = px.box(
        df,
        y=col,
        title=f"Boxplot of {feat} ({col})",
        points="outliers",
        color_discrete_sequence=["#A23B72"],
    )
    fig_box.show()


**Discussion of possible outliers:**
- Points plotted individually beyond the whiskers in each boxplot represent potential outliers — values that
  are unusually high or low relative to the bulk of the data for that feature.
- For sensor data collected in the field, outliers can arise from **genuine extreme events** (e.g., an
  exceptionally heavy rainfall reading right before a landslide) or from **sensor malfunctions** (e.g., a
  faulty humidity sensor reporting an impossible value).
- Distinguishing between "real but rare" and "erroneous" outliers is an important judgment call that should be
  informed by domain knowledge (e.g., physically plausible ranges for each sensor type) — this judgment and any
  corrective action (capping, removal, transformation) is deliberately **left for the preprocessing phase** and
  not performed in this notebook.
- Features with a tall box (large interquartile range) indicate high natural variability in that measurement,
  while a very short box with many outlier points suggests the majority of readings are tightly clustered with
  a handful of extreme exceptions.


---
## 9. Correlation Analysis

Correlation analysis helps us understand linear relationships between numerical features. This is useful for
identifying features that move together (potential redundancy / multicollinearity) as well as features that
appear largely independent of one another. **No features are removed** based on this analysis — that decision,
if warranted, belongs to a later feature engineering or preprocessing phase.


In [ ]:
# Compute correlation matrix on numerical columns only
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
corr_matrix


In [ ]:
# Plotly correlation heatmap
fig_corr = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation Heatmap of Numerical Features",
)
fig_corr.update_layout(width=900, height=800)
fig_corr.show()


In [ ]:
# Identify strongly correlated feature pairs (excluding self-correlation)
corr_pairs = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["Feature 1", "Feature 2", "Correlation"]
corr_pairs["Abs Correlation"] = corr_pairs["Correlation"].abs()
corr_pairs = corr_pairs.sort_values(by="Abs Correlation", ascending=False)

print("Top 10 most correlated feature pairs (by absolute value):")
corr_pairs.head(10)


**Findings:**
- **Highly correlated features** (absolute correlation typically above ~0.7) appear at the top of the table
  above. Such pairs move strongly together and may carry overlapping (redundant) information — a signal of
  possible **multicollinearity** that would be relevant for linear models in later phases.
- **Weakly correlated features** (absolute correlation close to 0) show little to no linear relationship,
  meaning they likely provide relatively independent information to a future model.
- The heatmap's diagonal is always 1.0 (a feature perfectly correlates with itself) and is not informative.
- It is worth noting that correlation only captures **linear** relationships — two features could still be
  strongly related in a non-linear way while showing a low Pearson correlation coefficient here.
- As instructed, **no features are dropped or transformed** based on these findings in this notebook; this
  analysis is purely diagnostic and will inform decisions in later, separate phases of the project.


---
## 10. Feature vs Target Analysis

Here we examine how individual features vary with respect to the target variable, `Label`. This helps build
intuition about which features might be informative for distinguishing landslide from non-landslide
conditions. We use box/violin plots to compare distributions of a feature across the classes of `Label`.


In [ ]:
# Features of interest for feature-vs-target comparison
fvt_features = ["Rainfall", "Soil Moisture", "Humidity", "Slope Angle"]

for feat in fvt_features:
    col = feature_map.get(feat)
    if col is None or col not in df.select_dtypes(include=[np.number]).columns:
        print(f"Skipping '{feat}': no matching numeric column found.")
        continue

    fig_fvt = px.box(
        df,
        x="Label",
        y=col,
        color="Label",
        title=f"{feat} ({col}) vs Label",
        points="outliers",
    )
    fig_fvt.show()


**Observations:**
- **Rainfall vs Label:** if landslide events (positive class) show a noticeably higher median rainfall than
  non-events, this suggests rainfall is a strong candidate predictor of landslide occurrence — consistent with
  the well-established role of rainfall as a landslide trigger.
- **Soil Moisture vs Label:** elevated soil moisture in the landslide class would reinforce the hypothesis that
  saturated soil conditions precede slope failure.
- **Humidity vs Label:** humidity may show a weaker or more ambiguous relationship with `Label` compared to
  direct precipitation/soil measurements, since humidity is a more indirect environmental indicator.
- **Slope Angle vs Label:** since slope angle is a static topographic property, a clear separation between
  classes here would indicate that certain slope angles are inherently more prone to landslides, independent of
  transient weather conditions.
- These comparisons are exploratory and descriptive; no statistical significance testing or feature selection
  is performed at this stage.


---
## 11. Pair Plot

A pair plot lets us view pairwise relationships between several variables simultaneously, with the target
`Label` used to color the points. We select approximately six meaningful numerical variables (based on the
feature matches identified in Section 7) alongside `Label`.


In [ ]:
# Select approximately six meaningful variables for the pair plot
pairplot_feature_names = ["Rainfall", "Soil Moisture", "Temperature", "Humidity", "Slope Angle", "Elevation"]
pairplot_cols = [feature_map[f] for f in pairplot_feature_names if feature_map.get(f) is not None]
pairplot_cols = [c for c in pairplot_cols if c in df.select_dtypes(include=[np.number]).columns]

print("Columns selected for pair plot:", pairplot_cols)

pairplot_df = df[pairplot_cols + ["Label"]].copy()

fig_pair = px.scatter_matrix(
    pairplot_df,
    dimensions=pairplot_cols,
    color="Label",
    title="Pair Plot of Selected Numerical Features Colored by Label",
    opacity=0.6,
)
fig_pair.update_layout(width=1000, height=1000)
fig_pair.update_traces(diagonal_visible=False)
fig_pair.show()


**Observations:**
- The diagonal-free scatter matrix lets us visually scan for pairwise separations between the two `Label`
  classes across multiple feature combinations at once.
- Feature pairs where the two color groups (landslide vs. non-landslide) form visibly separated clusters are
  promising candidates for predictive power in later modeling phases.
- Feature pairs where the two color groups are heavily overlapping suggest that, on their own, those two
  dimensions may not strongly discriminate between classes — though combinations with other features (not shown
  here) could still be informative.
- This is a purely visual, exploratory tool; no feature selection decisions are finalized here.


---
## 12. Final Summary

### Key Findings

- **Dataset structure:** The dataset contains WSN sensor readings alongside a binary `Label` target column
  indicating landslide occurrence. Section 3 and 4 established the overall shape, column names, and data types.
- **Target balance:** Section 5 revealed the class distribution of `Label`. If the classes are imbalanced, this
  is an important consideration for evaluation metrics and modeling strategy in later phases.
- **Data completeness:** Section 6 quantified missing values per column. Any columns with non-trivial
  missingness will require a deliberate imputation or removal strategy during preprocessing (not performed
  here).
- **Feature distributions:** Section 7 showed that key environmental features (Rainfall, Soil Moisture,
  Temperature, Humidity, Slope Angle, Elevation, NDVI) exhibit a range of distribution shapes, from
  right-skewed (e.g., Rainfall) to more symmetric (e.g., Temperature).
- **Outliers:** Section 8's boxplots flagged potential outliers in several features. These may reflect genuine
  extreme environmental events or sensor errors, and should be investigated further during preprocessing.
- **Correlation structure:** Section 9 identified pairs of features with notably high or low linear
  correlation, hinting at possible multicollinearity that could matter for certain model types later on. No
  features were removed at this stage.
- **Feature–target relationships:** Section 10 suggested that features such as Rainfall and Soil Moisture may
  show visibly different distributions between landslide and non-landslide records, making them promising
  candidate predictors.
- **Multivariate view:** Section 11's pair plot offered a combined view of several key features colored by
  `Label`, helping to visually assess which feature combinations may separate the classes most clearly.

### Next Steps (Out of Scope for This Notebook)

The insights gathered in this EDA phase — regarding missing data, outliers, feature distributions, correlation,
and class balance — will directly inform the **next phase** of the GeoSlide project: data preprocessing
(cleaning, encoding, normalization/standardization, and dataset splitting), followed eventually by feature
engineering and model training. None of those steps have been performed in this notebook, consistent with the
Phase 2 scope of "Exploratory Data Analysis only."
